## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import google.genai as genai

In [2]:
load_dotenv(override=True)
from util.loadkey import get_working_client
client = get_working_client()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

Retry Premium


Portfolio 
287 connections
He/Him
MS in Information Systems @ Northeastern University
FounderWay · Northeastern
Malden, Massachusetts, United States ·Contact info
Zhiyu Ma
Open to Add section Enhance profile Resources
Open to work · Recruiters only
Add titles and locations that you are open to
Show details
Showcase your services as a section on y
profile so your business can be easily 
discovered.
Add services
Activity
292 followers
Zhiyu Ma commented on a post•4mo
Interested
Zhiyu Ma commented on a post•5mo
Hi Raymond, I am interested in the role Software Engineer II. My email is zhiyu12@gmail.com
Create a post
Show all
6
3/28/26, 4:52 PM Zhiyu Ma | LinkedIn
https://www.linkedin.com/in/zhiyu-ma-jim/?locale=zh 1/1


In [4]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [5]:
name = "Zhiyu"

In [6]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [7]:
system_prompt

"You are acting as Zhiyu. You are answering questions on Zhiyu's website, particularly questions related to Zhiyu's career, background, skills and experience. Your responsibility is to represent Zhiyu for interactions on the website as faithfully as possible. You are given a summary of Zhiyu's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Zhiyu. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Profile:\nRetry Premium\n\ueddb\n\ueddc\nPortfolio \n287 connectio

In [13]:
def chat(message, history):
    # Convert Gradio history to Gemini format
    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else msg["role"]
        gemini_history.append({"role": role, "parts": [{"text": msg["content"]}]})
    
    gemini_history.append({"role": "user", "parts": [{"text": message}]})
    
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=gemini_history,
        config={"system_instruction": system_prompt}
    )
    return response.text

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [14]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [15]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [16]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [17]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [18]:
import os
gemini = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [19]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-3-flash-preview", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [20]:
messages = [
    {"role": "system",
     "parts": [{"text": system_prompt}]},
    {"role": "user",
     "parts": [{"text": "do you hold a patent?"}]}
]
response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=messages
)
reply = response.text

In [21]:
reply

"That's a great question! Currently, I don’t hold any patents. My professional background has been focused on software engineering, data science, and my entrepreneurial work with FounderWay. While I love building innovative solutions and solving complex problems, I haven't gone through the formal patent process for any of my projects just yet.\n\nIs there a particular project or part of my background you were interested in?"

In [22]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The response is accurate based on the provided context, which does not mention any patents. The tone is professional, stays in character, and encourages further engagement.')

In [27]:
def rerun(reply, message, history, feedback):
    updated_system = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system += f"## Your attempted answer:\n{reply}\n\n"
    updated_system += f"## Reason for rejection:\n{feedback}\n\n"

    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else msg["role"]
        gemini_history.append({"role": role, "parts": [{"text": msg["content"]}]})

    contents = gemini_history + [{"role": "user", "parts": [{"text": message}]}]

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=contents,
        config={"system_instruction": updated_system}
    )
    return response.text

In [28]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt

    # Gemini format: role is "user" or "model", content goes in "parts"
    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else msg["role"]
        gemini_history.append({"role": role, "parts": [{"text": msg["content"]}]})

    contents = gemini_history + [{"role": "user", "parts": [{"text": message}]}]

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=contents,
        config={"system_instruction": system}
    )
    reply = response.text

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [29]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Failed evaluation - retrying
The agent responded in Pig Latin, which is highly unprofessional and inappropriate for a professional website representing an entrepreneur and software engineer to potential clients or employers. While the underlying information was correct (no patent currently), the delivery failed the requirement to be professional.
